In [3]:
import os
from dotenv import load_dotenv

load_dotenv()
assert os.getenv("GROQ_API_KEY"), "Set GROQ_API_KEY in a .env file or above"
print("API key loaded.")

API key loaded.


In [4]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "telecom_guide.pdf"  

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Loaded {len(pages)} pages from the PDF.")
print("\n--- First page preview (first 500 chars) ---")
print(pages[0].page_content[:500])

C:\Users\aimga\AppData\Local\Temp\ipykernel_15268\2192669542.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
D:\Agenticai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 9 pages from the PDF.

--- First page preview (first 500 chars) ---
Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,       # ~150 words per chunk
    chunk_overlap=100,    # overlap keeps context at boundaries
    separators=["\n\n", "\n", ".", " "],  # tries paragraph → line → sentence → word
)

chunks = splitter.split_documents(pages)

print(f"Total chunks: {len(chunks)}  (from {len(pages)} pages)")
print(f"Avg chunk length: {sum(len(c.page_content) for c in chunks) // len(chunks)} chars")
print("\n--- Example chunk ---")
print(chunks[5].page_content)

Total chunks: 37  (from 9 pages)
Avg chunk length: 504 chars

--- Example chunk ---
Telecom Technical Reference Guide  - Internal Use Only
2. Troubleshooting Connectivity Issues
Connectivity problems are the most common category of customer complaints. A structured diagnostic approach
resolves the majority of cases without escalation.
Step 1  - Verify signal strength. Open the device's status bar or dial *3001#12345#* (iOS) or use a network signal
app (Android) to view the raw signal level in dBm. A signal above -85 dBm is good; between -85 and -100 dBm is
marginal; below -100 dBm is poor. If signal is weak, moving closer to a window or to a higher floor often helps.


In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

print("Loading embedding model (downloads ~90 MB on first run)...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

print("Embedding all chunks and storing in Chroma (in memory)...")
vector_store = Chroma.from_documents(chunks, embeddings)

print(f"Vector store ready. {vector_store._collection.count()} vectors stored.")

Loading embedding model (downloads ~90 MB on first run)...


D:\Agenticai\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aimga\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3573.37it/s]


Embedding all chunks and storing in Chroma (in memory)...
Vector store ready. 37 vectors stored.


In [7]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

test_query = "What is VoLTE and how does it improve call quality?"
retrieved = retriever.invoke(test_query)

print(f"Query: {test_query}")
print(f"Retrieved {len(retrieved)} chunks:\n")
for i, doc in enumerate(retrieved, 1):
    print(f"--- Chunk {i} ---")
    print(doc.page_content[:300])
    print()

Query: What is VoLTE and how does it improve call quality?
Retrieved 3 chunks:

--- Chunk 1 ---
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls 

--- Chunk 2 ---
voice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an
account that has VoLTE activated.
Enabling VoLTE: On most Android devices navigate to Settings > Mobile Network > VoLTE and toggle it on. On
iPhone go to Settings > Mobile Data > Mobile Data Opt

--- Chunk 3 ---
prioritised over general data traffic. This prevents voice quality degradation during periods of network congestion.
Without QoS, voice packets would compete with video streaming and file downloads, causing jitter and packet
loss.
Fallback Behaviour: If a VoLTE call c

In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

# --- Helper: join retrieved chunks into a single context string ---
def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)


# --- System prompt: ground the LLM in the retrieved context ---
SYSTEM_PROMPT = """\
You are a helpful telecom assistant.
Answer the question using ONLY the context provided below.
If the context does not contain enough information, say so clearly.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"),
])

# --- LLM via Groq API ---
llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    reasoning_format="parsed",
)

# --- Assemble the chain ---
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain assembled.")

RAG chain assembled.


In [9]:
question = "How does international roaming work and what charges should I expect?"

print(f"Q: {question}\n")
print("A:", chain.invoke(question))

Q: How does international roaming work and what charges should I expect?

A: International roaming allows your device to connect to partner networks abroad. Here’s how it works and the charges you may incur:

### **How It Works**  
1. **Authentication**: The visited network verifies your subscription via signaling protocols (SS7/Diameter), and your home network authorizes service.  
2. **Traffic Handling**: All data, voice, and SMS are routed back to your home network for billing, which introduces latency compared to local networks.  

### **Charges by Roaming Zone**  
- **Zone A** (EU, UK, Australia, New Zealand): Lowest rates.  
- **Zone B** (USA, Canada, Japan, Singapore): Moderate rates.  
- **Zone C** (Rest of the world): Highest per-MB/minute charges.  

### **Key Tips**  
- **Bundles**: Purchase a roaming bundle before traveling to Zone B/C to avoid high standard rates. Note: Pre-bundle usage is billed at standard rates and is not reversed.  
- **Coverage**: Check the provider’s

In [10]:
print("Telecom RAG Assistant — type 'quit' to exit\n")

while True:
    question = input("Your question: ").strip()
    if question.lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break
    if not question:
        continue

    print("\nAnswer:")
    for chunk in chain.stream(question):
        print(chunk, end="", flush=True)
    print("\n")

Telecom RAG Assistant — type 'quit' to exit



Your question:  What is the difference between 4G and 5G?



Answer:
The key differences between 4G and 5G are:  

1. **Speed**:  
   - 4G (LTE) offers typical download speeds of **20–150 Mbps**, with LTE-Advanced pushing speeds beyond **300 Mbps**.  
   - 5G targets **peak speeds of 10 Gbps**, a significant leap in performance.  

2. **Latency**:  
   - 4G has latency **below 50 ms**.  
   - 5G reduces latency to **under 1 ms**, critical for real-time applications.  

3. **Device Density**:  
   - 4G supports standard mobile connectivity.  
   - 5G can connect **up to 1 million devices per square kilometre**, enabling dense IoT deployments.  

4. **Spectrum Usage**:  
   - 4G operates primarily on frequencies like 700 MHz–2.6 GHz.  
   - 5G uses **sub-6 GHz** and **millimetre-wave (mmWave)** bands, allowing higher data capacity but requiring more infrastructure for coverage.  

5. **Use Cases**:  
   - 4G focuses on smartphone and mobile broadband.  
   - 5G supports **autonomous vehicles, smart cities, and massive IoT** due to its low latency

Your question:  How do I fix a SIM card that is not being detected?



Answer:
To fix a SIM card that is not being detected, follow these steps based on the provided context:

1. **Inspect the SIM (Step 4):**  
   - Power off the device.  
   - Remove the SIM card, clean its gold contacts with a dry cloth, and reseat it firmly. A partially seated or dirty SIM can cause detection issues.  

2. **Test with Another Device (Step 6):**  
   - Insert the SIM into a different handset to determine if the issue is with the SIM or the original device.  
     - If the SIM works in another device, the original device may have a hardware problem (e.g., damaged SIM slot or antenna) requiring repair.  
     - If the SIM still isn’t detected, the SIM itself may be faulty.  

3. **Check for Hardware Issues (Step 7):**  
   - If the problem persists, a faulty antenna or damaged SIM slot in the device could be the cause. Device repair may be necessary.  

No mention of network outages (Step 5) or SIM swap fraud (Page 6) is relevant here unless additional symptoms arise.



...


Answer:
**SIM Swap Fraud** occurs when a fraudster deceives a mobile carrier into transferring your phone number to a SIM card they control. This allows them to intercept SMS-based two-factor authentication (2FA) codes and gain access to your accounts.  

**How to Protect Yourself:**  
1. **Enable Strong Authentication**: Use app-based 2FA (e.g., authenticator apps) instead of SMS for critical accounts.  
2. **Secure Your Carrier Account**:  
   - Confirm your mobile carrier requires **government-issued ID** for in-person SIM replacements.  
   - For remote requests, ensure they use **multi-factor identity verification** (e.g., security questions, biometrics).  
3. **Monitor for Suspicious Activity**:  
   - If your carrier flags your account for a recent SIM swap, enable **elevated fraud monitoring** (typically 30 days post-swap).  
   - Watch for unexpected account changes (e.g., plan downgrades, address updates) and report them immediately.  
4. **Stay Vigilant**: Avoid sharing per

Your question:  Explain VoLTE in simple terms.



Answer:
VoLTE (Voice over LTE) is a technology that lets you make voice calls over a 4G LTE network instead of the older 2G/3G networks. Here’s the simple breakdown:  

- **How it works**: It sends your voice as data packets over the internet (like a video call) instead of using the old "circuit-switched" method.  
- **Benefits**:  
  - **Clearer calls**: HD voice quality (like a higher-fidelity music stream).  
  - **Faster call setup**: Calls connect in under 2 seconds (vs. 6-8 seconds on older networks).  
  - **Use data while talking**: You can browse or stream during a call.  
- **Behind the scenes**: The network prioritizes voice to avoid dropouts, even if you’re using data. If LTE isn’t available, it automatically falls back to 3G/2G, but with lower quality.  

You need a compatible phone, a VoLTE-enabled SIM, and the feature activated by your carrier.



Your question:  quit


Goodbye!


In [11]:
print("Telecom RAG Assistant — type 'quit' to exit\n")

while True:
    question = input("Your question: ").strip()
    if question.lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break
    if not question:
        continue

    print("\nAnswer:")
    for chunk in chain.stream(question):
        print(chunk, end="", flush=True)
    print("\n")

Telecom RAG Assistant — type 'quit' to exit



Your question:  quit


Goodbye!


In [12]:
debug_question = "What security measures protect against SIM swap fraud?"

docs = retriever.invoke(debug_question)
print(f"Question: {debug_question}")
print(f"Retrieved {len(docs)} chunks:\n")
for i, doc in enumerate(docs, 1):
    print(f"{'='*60}")
    print(f"Chunk {i} (page {doc.metadata.get('page', '?')})")
    print(f"{'='*60}")
    print(doc.page_content)
    print()

print("\nFinal Answer:")
print(chain.invoke(debug_question))

Question: What security measures protect against SIM swap fraud?
Retrieved 3 chunks:

Chunk 1 (page 8)
mitigate this with SS7 firewalls and anomaly detection systems, but the risk cannot be fully eliminated on legacy
protocols. 5G's use of HTTPS-based APIs (Service Based Architecture) substantially reduces this attack surface.
SIM Swap Fraud: Described in Section 5. Key mitigation: enforce strict in-person or multi-factor remote identity
verification before any SIM replacement. Flag accounts with recent SIM swaps for elevated fraud monitoring for
30 days.
International Revenue Share Fraud (IRSF): Fraudsters compromise a PBX or customer account and generate

Chunk 2 (page 5)
provide an additional layer of protection; after three incorrect PIN attempts the SIM is locked and requires a PUK
code to unlock.
SIM Swap Fraud: SIM swap attacks occur when a fraudster convinces a carrier to transfer a victim's number to a
new SIM. This allows the attacker to intercept SMS-based two-factor authent